In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr
from tqdm import tqdm
pd.options.display.max_columns = 1000

In [ ]:
df = pd.read_csv(r'../data/05_model_input/AR6_scenarios_carbon_price_fixed.csv')
df.head()

In [ ]:
# Filter for years 2023-2030 and Power sector only
df_filtered = df[(df['scenario_year'] >= 2023) & 
                 (df['scenario_year'] <= 2030) & 
                 (df['sector'] == 'Power')].copy()

print(f"Filtered data shape: {df_filtered.shape}")
print(f"Years: {sorted(df_filtered['scenario_year'].unique())}")
print(f"Number of scenario providers: {df_filtered['scenario_provider'].nunique()}")


In [ ]:
# Variables to compare
variables_to_compare = [
    'scenario_price', 
    'fuel_price', 
    'scenario_pathway', 
    'scenario_capacity_factor', 
    'efficiency_decimal'
]

def calculate_trajectory_distance(series1, series2):
    """
    Calculate distance between two time series.
    Returns multiple metrics: RMSE, mean absolute difference, and correlation.
    """
    # Remove NaN values where both series have valid data
    valid_mask = ~(pd.isna(series1) | pd.isna(series2))
    s1 = series1[valid_mask]
    s2 = series2[valid_mask]
    
    if len(s1) < 2:
        return np.nan, np.nan, np.nan
    
    # Calculate RMSE (root mean squared error)
    rmse = np.sqrt(np.mean((s1 - s2) ** 2))
    
    # Calculate mean absolute difference
    mad = np.mean(np.abs(s1 - s2))
    
    # Calculate correlation (for shape similarity)
    try:
        corr, _ = pearsonr(s1, s2)
    except:
        corr = np.nan
    
    return rmse, mad, corr

def calculate_normalized_distance(group1, group2, variables):
    """
    Calculate normalized distance across multiple variables.
    Returns a dict with distances for each variable and an overall score.
    """
    distances = {}
    
    for var in variables:
        rmse, mad, corr = calculate_trajectory_distance(
            group1[var].values, 
            group2[var].values
        )
        
        # Normalize by mean to make comparable across variables
        mean_val = (group1[var].mean() + group2[var].mean()) / 2
        if mean_val != 0 and not pd.isna(mean_val):
            normalized_mad = mad / abs(mean_val)
        else:
            normalized_mad = mad
            
        distances[f'{var}_rmse'] = rmse
        distances[f'{var}_mad'] = mad
        distances[f'{var}_normalized_mad'] = normalized_mad
        distances[f'{var}_correlation'] = corr
    
    # Overall distance: average of normalized MADs (lower is better)
    valid_norm_mads = [distances[f'{var}_normalized_mad'] 
                       for var in variables 
                       if not pd.isna(distances[f'{var}_normalized_mad'])]
    
    distances['overall_distance'] = np.mean(valid_norm_mads) if valid_norm_mads else np.nan
    
    return distances

print("Functions defined successfully!")


In [ ]:
(df
.loc[df["sector"]=="Power",["scenario_provider", "scenario", "technology"]]
.drop_duplicates()
.groupby(["scenario_provider", "scenario", "technology"])
)

In [ ]:
df["scenario_provider"].unique()

In [ ]:
df_filtered = df_filtered[
    df_filtered["scenario_provider"] == "REMIND-MAgPIE 2.1-4.3"
    ]

In [ ]:
# Calculate pairwise distances for all scenario pairs within each provider+geography+technology
results = []

# Group by provider, geography, and technology
grouped = df_filtered.groupby(['scenario_provider', 'scenario_geography', 'technology'])

total_groups = len(grouped)
print(f"Processing {total_groups} provider+geography+technology combinations...")

for (provider, geography, technology), group_data in tqdm(grouped, total=total_groups, desc="Processing groups"):
    # Get unique scenarios in this group
    scenarios = group_data['scenario'].unique()
    
    # Only process if there are at least 2 scenarios to compare
    if len(scenarios) < 2:
        continue
    
    # Compare all pairs of scenarios
    for scenario1, scenario2 in combinations(scenarios, 2):
        # Get data for each scenario (sorted by year)
        data1 = group_data[group_data['scenario'] == scenario1].sort_values('scenario_year')
        data2 = group_data[group_data['scenario'] == scenario2].sort_values('scenario_year')
        
        # Make sure they have the same years
        common_years = set(data1['scenario_year']) & set(data2['scenario_year'])
        if len(common_years) < 2:  # Need at least 2 years to compare
            continue
        
        data1 = data1[data1['scenario_year'].isin(common_years)]
        data2 = data2[data2['scenario_year'].isin(common_years)]
        
        # Calculate distances
        distances = calculate_normalized_distance(data1, data2, variables_to_compare)
        
        # Store result
        result = {
            'scenario_provider': provider,
            'scenario_geography': geography,
            'technology': technology,
            'scenario_1': scenario1,
            'scenario_2': scenario2,
            'n_years_compared': len(common_years),
            **distances
        }
        results.append(result)

# Convert to DataFrame
df_distances = pd.DataFrame(results)

print(f"\nCompleted! Generated {len(df_distances)} pairwise comparisons.")
print(f"Scenario providers: {df_distances['scenario_provider'].nunique()}")
df_distances.head()


In [ ]:
# Summary statistics
print("Summary of overall distances (lower is better, means trajectories are more similar):")
print(df_distances['overall_distance'].describe())

print("\n\nMost similar trajectories (lowest overall distance):")
print(df_distances.nsmallest(10, 'overall_distance')[
    ['scenario_provider', 'scenario_geography', 'technology', 
     'scenario_1', 'scenario_2', 'overall_distance']
])


In [ ]:
print("Most dissimilar trajectories (highest overall distance):")
print(df_distances.nlargest(10, 'overall_distance')[
    ['scenario_provider', 'scenario_geography', 'technology', 
     'scenario_1', 'scenario_2', 'overall_distance']
])


In [ ]:
# Average distance by scenario provider
print("\nAverage trajectory distance by scenario provider:")
provider_summary = df_distances.groupby('scenario_provider').agg({
    'overall_distance': ['mean', 'median', 'std', 'count']
}).round(3)
provider_summary.columns = ['mean_distance', 'median_distance', 'std_distance', 'n_comparisons']
print(provider_summary.sort_values('mean_distance'))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)


In [ ]:
def plot_scenario_comparison(provider, scenario_1, scenario_2, 
                            variable='scenario_pathway', 
                            df_source=None,
                            save_path=None):
    """
    Plot comparison of two scenarios across all geographies and technologies.
    
    Parameters:
    -----------
    provider : str
        Scenario provider name
    scenario_1 : str
        First scenario name
    scenario_2 : str
        Second scenario name
    variable : str
        Variable to plot (default: 'scenario_pathway')
        Options: 'scenario_pathway', 'scenario_price', 'fuel_price', 
                 'scenario_capacity_factor', 'efficiency_decimal'
    df_source : pd.DataFrame
        Source dataframe (if None, uses df_filtered from global scope)
    save_path : str, optional
        Path to save the plot
    
    Returns:
    --------
    fig : matplotlib figure
    """
    if df_source is None:
        df_source = df_filtered
    
    # Filter data for the two scenarios
    data = df_source[
        (df_source['scenario_provider'] == provider) &
        (df_source['scenario'].isin([scenario_1, scenario_2]))
    ].copy()
    
    if len(data) == 0:
        print(f"No data found for provider={provider}, scenarios={scenario_1}, {scenario_2}")
        return None
    
    # Get distance info if available
    distance_info = None
    if 'df_distances' in globals():
        dist_row = df_distances[
            (df_distances['scenario_provider'] == provider) &
            (((df_distances['scenario_1'] == scenario_1) & (df_distances['scenario_2'] == scenario_2)) |
             ((df_distances['scenario_1'] == scenario_2) & (df_distances['scenario_2'] == scenario_1)))
        ]
        if len(dist_row) > 0:
            # Average distance across all geo+tech combinations
            distance_info = dist_row['overall_distance'].mean()
    
    # Get unique technologies and geographies
    technologies = sorted(data['technology'].unique())
    geographies = sorted(data['scenario_geography'].unique())
    
    # Create subplots - one row per technology, one column per geography
    n_techs = len(technologies)
    n_geos = len(geographies)
    
    # Adjust layout based on number of subplots
    if n_geos <= 3:
        ncols = n_geos
        nrows = n_techs
        figsize = (6 * ncols, 4 * nrows)
    else:
        # If too many geographies, limit columns
        ncols = min(4, n_geos)
        nrows = n_techs * ((n_geos + ncols - 1) // ncols)
        figsize = (6 * ncols, 4 * nrows)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    
    # Flatten axes for easier iteration if needed
    if n_geos <= 3:
        # Standard layout: technologies as rows, geographies as columns
        for i, tech in enumerate(technologies):
            for j, geo in enumerate(geographies):
                ax = axes[i, j]
                
                # Filter data for this tech and geo
                subset = data[
                    (data['technology'] == tech) &
                    (data['scenario_geography'] == geo)
                ].sort_values('scenario_year')
                
                if len(subset) > 0:
                    # Plot each scenario
                    for scenario in [scenario_1, scenario_2]:
                        scenario_data = subset[subset['scenario'] == scenario]
                        if len(scenario_data) > 0:
                            ax.plot(scenario_data['scenario_year'], 
                                   scenario_data[variable],
                                   marker='o', 
                                   label=scenario,
                                   linewidth=2,
                                   markersize=6)
                    
                    # Formatting
                    ax.set_xlabel('Year', fontsize=10)
                    ax.set_ylabel(variable.replace('_', ' ').title(), fontsize=10)
                    ax.set_title(f"{tech} - {geo}", fontsize=11, fontweight='bold')
                    ax.legend(fontsize=8, loc='best')
                    ax.grid(True, alpha=0.3)
                else:
                    ax.text(0.5, 0.5, 'No data', 
                           ha='center', va='center', 
                           transform=ax.transAxes)
                    ax.set_title(f"{tech} - {geo}", fontsize=11)
    else:
        # Alternative layout for many geographies
        plot_idx = 0
        for tech in technologies:
            for geo in geographies:
                row = plot_idx // ncols
                col = plot_idx % ncols
                ax = axes[row, col]
                
                # Filter data for this tech and geo
                subset = data[
                    (data['technology'] == tech) &
                    (data['scenario_geography'] == geo)
                ].sort_values('scenario_year')
                
                if len(subset) > 0:
                    # Plot each scenario
                    for scenario in [scenario_1, scenario_2]:
                        scenario_data = subset[subset['scenario'] == scenario]
                        if len(scenario_data) > 0:
                            ax.plot(scenario_data['scenario_year'], 
                                   scenario_data[variable],
                                   marker='o', 
                                   label=scenario,
                                   linewidth=2,
                                   markersize=6)
                    
                    # Formatting
                    ax.set_xlabel('Year', fontsize=10)
                    ax.set_ylabel(variable.replace('_', ' ').title(), fontsize=10)
                    ax.set_title(f"{tech} - {geo}", fontsize=11, fontweight='bold')
                    ax.legend(fontsize=8, loc='best')
                    ax.grid(True, alpha=0.3)
                else:
                    ax.text(0.5, 0.5, 'No data', 
                           ha='center', va='center', 
                           transform=ax.transAxes)
                    ax.set_title(f"{tech} - {geo}", fontsize=11)
                
                plot_idx += 1
        
        # Hide unused subplots
        for idx in range(plot_idx, nrows * ncols):
            row = idx // ncols
            col = idx % ncols
            axes[row, col].axis('off')
    
    # Main title
    title = f"{provider}\nComparing: {scenario_1} vs {scenario_2}"
    if distance_info is not None:
        title += f"\n(Avg Overall Distance: {distance_info:.4f})"
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")
    
    return fig

print("Plotting function defined successfully!")


## Iterative Pair Selection

Find interesting scenario pairs by excluding uninteresting ones (e.g., baseline scenarios):


In [ ]:
# Interactive pair selector - refine by rejecting unwanted scenarios
def interactive_pair_selector(provider, df_dist=None):
    """
    Interactive selector to find the best pair by iteratively rejecting scenarios.
    
    Usage: 
        interactive_pair_selector("REMIND-MAgPIE 2.1-4.3")
    
    Then follow the prompts:
        - Enter 1: Keep scenario 1, find next best match for it
        - Enter 2: Keep scenario 2, find next best match for it  
        - Enter 3: Accept current pair and plot
    """
    if df_dist is None:
        df_dist = df_distances
    
    # Track rejected scenarios
    rejected_scenarios = []
    
    # Get initial closest pair for this provider
    provider_data = df_dist[df_dist['scenario_provider'] == provider]
    
    if len(provider_data) == 0:
        print(f"No data found for provider: {provider}")
        return None
    
    # Calculate average distance for all pairs (across all geo+tech)
    pair_distances = provider_data.groupby(['scenario_1', 'scenario_2']).agg({
        'overall_distance': 'mean',
        'n_years_compared': 'mean'
    }).reset_index().sort_values('overall_distance')
    
    if len(pair_distances) == 0:
        print(f"No scenario pairs found for provider: {provider}")
        return None
    
    # Start with the closest pair
    current_pair = pair_distances.iloc[0]
    scenario_1 = current_pair['scenario_1']
    scenario_2 = current_pair['scenario_2']
    distance = current_pair['overall_distance']
    
    iteration = 1
    
    while True:
        print(f"\n{'='*80}")
        print(f"ITERATION {iteration} - Provider: {provider}")
        print(f"{'='*80}")
        print(f"\nCurrent best pair (avg distance: {distance:.4f}):")
        print(f"  [1] {scenario_1}")
        print(f"  [2] {scenario_2}")
        print(f"\nOptions:")
        print(f"  1 - Keep scenario [1], find next best match")
        print(f"  2 - Keep scenario [2], find next best match")
        print(f"  3 - Accept this pair and plot comparison")
        print(f"  q - Quit without plotting")
        
        if len(rejected_scenarios) > 0:
            print(f"\nRejected scenarios so far: {len(rejected_scenarios)}")
            print(f"  {', '.join(rejected_scenarios[:5])}" + 
                  (f" ... and {len(rejected_scenarios)-5} more" if len(rejected_scenarios) > 5 else ""))
        
        choice = input("\nYour choice: ").strip().lower()
        
        if choice == 'q':
            print("Cancelled.")
            return None
        
        if choice == '3':
            print(f"\n✓ Final selection: {scenario_1} vs {scenario_2}")
            print(f"  Distance: {distance:.4f}")
            print("\nGenerating plot...")
            
            fig = plot_scenario_comparison(
                provider=provider,
                scenario_1=scenario_1,
                scenario_2=scenario_2,
                variable='scenario_pathway'
            )
            plt.show()
            
            return {
                'provider': provider,
                'scenario_1': scenario_1,
                'scenario_2': scenario_2,
                'distance': distance,
                'iterations': iteration,
                'rejected': rejected_scenarios
            }
        
        elif choice in ['1', '2']:
            # Keep the chosen scenario, reject the other
            if choice == '1':
                keep_scenario = scenario_1
                reject_scenario = scenario_2
            else:
                keep_scenario = scenario_2
                reject_scenario = scenario_1
            
            rejected_scenarios.append(reject_scenario)
            print(f"\n→ Keeping: {keep_scenario}")
            print(f"\n→ Rejecting: {reject_scenario}")
            print(f"→ Finding next best match for '{keep_scenario}'...")
            
            # Find pairs involving the kept scenario
            relevant_pairs = provider_data[
                (provider_data['scenario_1'] == keep_scenario) | 
                (provider_data['scenario_2'] == keep_scenario)
            ].copy()
            
            # Get the "other" scenario
            relevant_pairs['other_scenario'] = relevant_pairs.apply(
                lambda row: row['scenario_2'] if row['scenario_1'] == keep_scenario else row['scenario_1'],
                axis=1
            )
            
            # Filter out rejected scenarios
            relevant_pairs = relevant_pairs[~relevant_pairs['other_scenario'].isin(rejected_scenarios)]
            
            if len(relevant_pairs) == 0:
                print(f"\n⚠ No more scenarios available to pair with '{keep_scenario}'")
                print("Returning to previous state...")
                rejected_scenarios.remove(reject_scenario)
                continue
            
            # Calculate average distance for each candidate
            candidate_distances = relevant_pairs.groupby('other_scenario').agg({
                'overall_distance': 'mean',
                'n_years_compared': 'mean'
            }).reset_index().sort_values('overall_distance')
            
            # Get the best match
            best_match = candidate_distances.iloc[0]
            new_scenario = best_match['other_scenario']
            new_distance = best_match['overall_distance']
            
            # Update current pair
            scenario_1 = keep_scenario
            scenario_2 = new_scenario
            distance = new_distance
            iteration += 1
            
        else:
            print("Invalid choice. Please enter 1, 2, 3, or q.")

print("Interactive pair selector ready!")
print("\nUsage:")
print('  result = interactive_pair_selector("REMIND-MAgPIE 2.1-4.3")')


## Interactive Pair Selection

Run the cell below to interactively select scenario pairs:


# Interactive selection for REMIND-MAgPIE 2.1-4.3
result = interactive_pair_selector("REMIND-MAgPIE 2.1-4.3")


In [ ]:
# Step-by-step pair selector (works better in Jupyter)
def step_by_step_selector(provider, rejected_scenarios=None, df_dist=None):
    """
    Step-by-step selector that shows options and returns data for next step.
    Use this instead of the interactive version in Jupyter.
    """
    if df_dist is None:
        df_dist = df_distances
    
    if rejected_scenarios is None:
        rejected_scenarios = []
    
    # Get data for this provider
    provider_data = df_dist[df_dist['scenario_provider'] == provider]
    
    if len(provider_data) == 0:
        print(f"No data found for provider: {provider}")
        return None
    
    # Calculate average distance for all pairs
    pair_distances = provider_data.groupby(['scenario_1', 'scenario_2']).agg({
        'overall_distance': 'mean',
        'n_years_compared': 'mean'
    }).reset_index().sort_values('overall_distance')
    
    # Filter out pairs containing rejected scenarios
    if rejected_scenarios:
        mask = ~(pair_distances['scenario_1'].isin(rejected_scenarios) | 
                 pair_distances['scenario_2'].isin(rejected_scenarios))
        pair_distances = pair_distances[mask]
    
    if len(pair_distances) == 0:
        print(f"No scenario pairs available after rejecting: {rejected_scenarios}")
        return None
    
    # Show top 10 options
    print(f"Provider: {provider}")
    print(f"Rejected scenarios: {rejected_scenarios}")
    print(f"\nTop 10 closest scenario pairs:")
    print("="*100)
    
    for i, (_, row) in enumerate(pair_distances.head(10).iterrows()):
        print(f"{i+1:2d}. {row['scenario_1']}")
        print(f"    {row['scenario_2']}")
        print(f"    Distance: {row['overall_distance']:.4f}")
        print()
    
    return pair_distances.head(10).copy()

def continue_with_choice(provider, choice, rejected_scenarios, pair_distances):
    """
    Continue the selection process based on user choice.
    
    Parameters:
    -----------
    provider : str
        Provider name
    choice : str
        Either 'scenario_1', 'scenario_2', or 'accept'
    rejected_scenarios : list
        List of previously rejected scenarios
    pair_distances : pd.DataFrame
        Current pair options
    """
    if choice == 'accept':
        # Accept the best pair and plot
        best_pair = pair_distances.iloc[0]
        scenario_1 = best_pair['scenario_1']
        scenario_2 = best_pair['scenario_2']
        distance = best_pair['overall_distance']
        
        print(f"✓ Final selection: {scenario_1} vs {scenario_2}")
        print(f"  Distance: {distance:.4f}")
        print("\nGenerating plot...")
        
        fig = plot_scenario_comparison(
            provider=provider,
            scenario_1=scenario_1,
            scenario_2=scenario_2,
            variable='scenario_pathway'
        )
        plt.show()
        
        return {
            'provider': provider,
            'scenario_1': scenario_1,
            'scenario_2': scenario_2,
            'distance': distance,
            'rejected': rejected_scenarios
        }
    
    else:
        # Keep chosen scenario, reject the other
        best_pair = pair_distances.iloc[0]
        if choice == 'scenario_1':
            keep_scenario = best_pair['scenario_1']
            reject_scenario = best_pair['scenario_2']
        else:  # choice == 'scenario_2'
            keep_scenario = best_pair['scenario_2']
            reject_scenario = best_pair['scenario_1']
        
        new_rejected = rejected_scenarios + [reject_scenario]
        
        print(f"→ Keeping: {keep_scenario}")
        print(f"→ Rejecting: {reject_scenario}")
        print(f"→ Finding next best match for '{keep_scenario}'...")
        
        # Continue with new rejected list
        return step_by_step_selector(provider, new_rejected)

print("Step-by-step selector ready!")
print("\nUsage:")
print('  # Step 1: See initial options')
print('  options = step_by_step_selector("REMIND-MAgPIE 2.1-4.3")')
print('')
print('  # Step 2: Make a choice and continue')
print('  options = continue_with_choice("REMIND-MAgPIE 2.1-4.3", "scenario_1", [], options)')
print('')
print('  # Step 3: Continue until you accept')
print('  result = continue_with_choice("REMIND-MAgPIE 2.1-4.3", "accept", rejected_scenarios, options)')


## Step-by-Step Pair Selection (Jupyter-friendly)

This approach works better in Jupyter notebooks:


In [ ]:
# Step 1: See initial options for REMIND-MAgPIE 2.1-4.3
options = step_by_step_selector("REMIND-MAgPIE 2.1-4.3")


In [ ]:
# Step 2: Make your choice and continue
# Uncomment one of these lines based on your choice:

# Keep scenario 1, reject scenario 2
# options = continue_with_choice("REMIND-MAgPIE 2.1-4.3", "scenario_1", [], options)

# Keep scenario 2, reject scenario 1  
# options = continue_with_choice("REMIND-MAgPIE 2.1-4.3", "scenario_2", [], options)

# Accept the current pair and plot
# result = continue_with_choice("REMIND-MAgPIE 2.1-4.3", "accept", [], options)

print("Uncomment one of the lines above to make your choice!")


In [ ]:
# Continue the selection process...
# After running the previous cell, you can continue rejecting scenarios:

# Example: If you rejected some scenarios, you can continue like this:
# rejected_scenarios = ["Scenario_You_Rejected"]  # Add scenarios you've rejected
# options = step_by_step_selector("REMIND-MAgPIE 2.1-4.3", rejected_scenarios)

print("Use this cell to continue the selection process after rejecting scenarios")


In [ ]:
# View the selection result
if result:
    print("\nSelection Summary:")
    print(f"  Provider: {result['provider']}")
    print(f"  Final pair: {result['scenario_1']} vs {result['scenario_2']}")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Iterations: {result['iterations']}")
    print(f"  Rejected scenarios: {len(result['rejected'])}")


## Try other providers

List of available providers to explore:


In [ ]:
# Show all available providers
print("Available scenario providers:")
print("="*80)
for i, provider in enumerate(sorted(df_distances['scenario_provider'].unique()), 1):
    n_scenarios = len(df_distances[df_distances['scenario_provider'] == provider]['scenario_1'].unique()) + \
                  len(df_distances[df_distances['scenario_provider'] == provider]['scenario_2'].unique())
    n_scenarios = len(set(df_distances[df_distances['scenario_provider'] == provider]['scenario_1'].unique()) | 
                      set(df_distances[df_distances['scenario_provider'] == provider]['scenario_2'].unique()))
    print(f"{i:2d}. {provider} ({n_scenarios} scenarios)")
    
print("\n" + "="*80)
print("To explore a provider, run:")
print('  result = interactive_pair_selector("PROVIDER_NAME")')


In [ ]:
# Step 3: Continue excluding until you find an interesting pair
# You can exclude just one scenario from a pair if you want to keep the other

# Example: Maybe the new top pair has one baseline scenario - exclude just that one
# excluded.append("SomeBaselineScenario")  # Add to existing exclusions
# result_3 = show_next_closest_pair(example_provider, excluded, top_n=5)

print("\nWorkflow:")
print("1. Run show_next_closest_pair() to see top pairs")
print("2. Identify uninteresting scenarios (e.g., baselines, no-policy scenarios)")
print("3. Add them to the excluded list")
print("4. Run show_next_closest_pair() again with updated exclusions")
print("5. Repeat until you find an interesting pair (scenarios that should differ but don't)")


### Template for Manual Exploration

Use this cell to manually explore scenario pairs for any provider:


In [ ]:
# Template for manual exploration
# Uncomment and modify:

# # Choose your provider
# my_provider = "REMIND 3.0"  # Change this
# 
# # Start with no exclusions
# my_excluded = []
# 
# # Step 1: See initial options
# pairs = show_next_closest_pair(my_provider, my_excluded, top_n=10)
# 
# # Step 2: Exclude uninteresting scenarios
# # (e.g., both are baseline/no-policy scenarios)
# my_excluded = ["Scenario_Baseline", "Scenario_NoPolicy"]  # Add scenario names
# 
# # Step 3: See next options
# pairs = show_next_closest_pair(my_provider, my_excluded, top_n=10)
# 
# # Step 4: Keep excluding until you find interesting pairs
# # my_excluded.append("Another_Uninteresting_Scenario")
# # pairs = show_next_closest_pair(my_provider, my_excluded, top_n=10)

print("Template ready - uncomment and modify above to explore!")


### Batch Processing: Find Best Pairs for All Providers with Keyword Filters

Automatically find interesting pairs by excluding scenarios with certain keywords (e.g., "baseline", "reference"):


In [ ]:
def find_interesting_pairs_all_providers(exclude_keywords=None, df_dist=None):
    """
    Find closest pairs for all providers, excluding scenarios with certain keywords.
    
    Parameters:
    -----------
    exclude_keywords : list, optional
        Keywords to identify uninteresting scenarios (e.g., ['baseline', 'reference', 'nopolicy'])
        Scenarios containing these keywords (case-insensitive) will be excluded
    df_dist : pd.DataFrame, optional
        Distance dataframe
    
    Returns:
    --------
    pd.DataFrame with best pairs per provider
    """
    if df_dist is None:
        df_dist = df_distances
    
    if exclude_keywords is None:
        exclude_keywords = []
    
    results = []
    
    for provider in tqdm(df_dist['scenario_provider'].unique(), desc="Processing providers"):
        # Get all scenarios for this provider
        provider_scenarios = set(
            list(df_dist[df_dist['scenario_provider'] == provider]['scenario_1'].unique()) +
            list(df_dist[df_dist['scenario_provider'] == provider]['scenario_2'].unique())
        )
        
        # Identify scenarios to exclude based on keywords
        excluded = []
        if exclude_keywords:
            for scenario in provider_scenarios:
                scenario_lower = scenario.lower()
                if any(keyword.lower() in scenario_lower for keyword in exclude_keywords):
                    excluded.append(scenario)
        
        # Find closest pair excluding these scenarios
        result = find_closest_pair_excluding(provider, excluded, df_dist)
        
        if result:
            results.append({
                'scenario_provider': provider,
                'scenario_1': result['scenario_1'],
                'scenario_2': result['scenario_2'],
                'avg_overall_distance': result['avg_overall_distance'],
                'n_scenarios_excluded': len(excluded),
                'excluded_keywords': ','.join(exclude_keywords) if exclude_keywords else 'None'
            })
    
    df_results = pd.DataFrame(results).sort_values('avg_overall_distance')
    return df_results


# Example: Find interesting pairs, excluding scenarios with "baseline" or "reference" in name
print("Finding interesting pairs (excluding baseline/reference scenarios)...\n")

interesting_pairs = find_interesting_pairs_all_providers(
    exclude_keywords=['baseline', 'reference', 'nopolicy', 'historical']
)

print(f"\nFound {len(interesting_pairs)} providers with interesting pairs")
print("\nTop 10 providers with most similar 'interesting' scenario pairs:")
print(interesting_pairs.head(10))


In [ ]:
# Find the closest scenario pair for each provider (best overall proximity)
print("Finding closest scenario pairs per provider...\n")

closest_pairs = []
for provider in df_distances['scenario_provider'].unique():
    provider_data = df_distances[df_distances['scenario_provider'] == provider]
    
    # Group by scenario pair and calculate average distance across all geo+tech
    pair_distances = provider_data.groupby(['scenario_1', 'scenario_2']).agg({
        'overall_distance': 'mean',
        'n_years_compared': 'mean'
    }).reset_index()
    
    # Find the pair with minimum average distance
    if len(pair_distances) > 0:
        closest = pair_distances.loc[pair_distances['overall_distance'].idxmin()]
        closest_pairs.append({
            'scenario_provider': provider,
            'scenario_1': closest['scenario_1'],
            'scenario_2': closest['scenario_2'],
            'avg_overall_distance': closest['overall_distance'],
            'avg_n_years': closest['n_years_compared']
        })

df_closest_pairs = pd.DataFrame(closest_pairs).sort_values('avg_overall_distance')

print(f"Found {len(df_closest_pairs)} providers with comparable scenarios\n")
print("Top 10 providers with most similar scenario pairs:")
print(df_closest_pairs.head(10))


In [ ]:
# Average distance by technology across all providers
print("\nAverage trajectory distance by technology:")
tech_summary = df_distances.groupby('technology').agg({
    'overall_distance': ['mean', 'median', 'std', 'count']
}).round(3)
tech_summary.columns = ['mean_distance', 'median_distance', 'std_distance', 'n_comparisons']
print(tech_summary.sort_values('mean_distance'))


In [ ]:
# View detailed metrics for a specific provider (example)
if len(df_distances) > 0:
    example_provider = df_distances['scenario_provider'].iloc[0]
    print(f"\nDetailed view for provider: {example_provider}")
    provider_data = df_distances[df_distances['scenario_provider'] == example_provider]
    
    # Show column names available
    print(f"\nColumns with distance metrics:")
    metric_cols = [col for col in df_distances.columns if any(x in col for x in ['_mad', '_rmse', '_correlation', 'overall'])]
    print(metric_cols)
    
    # Show sample data
    print(f"\nSample comparisons for {example_provider}:")
    display_cols = ['scenario_geography', 'technology', 'scenario_1', 'scenario_2', 
                    'overall_distance', 'n_years_compared']
    print(provider_data[display_cols].head(10))
